# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Shehzadi434/flyrank-Internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


**Rule in plain words:**
"A page is worth reviewing for refresh if it has significant impressions, ranks well enough, but has low CTR and hasn't been updated recently."

**Score formula:**
baseline_score = impressions_90d × (1 - ctr_90d) × (content_age_days / 365)


**Reason codes:**
| Code | Condition | Action |
|------|-----------|--------|
| `high_impressions_low_ctr` | impressions > 500, ctr < 0.01 | Refresh content |
| `stale_visible_page` | content_age > 365, impressions > 500 | Update content |
| `position_opportunity` | avg_position > 10, impressions > 500 | Improve SEO |
| `other` | None of the above | Monitor only |

**Distribution (n=176,738):**
- `high_impressions_low_ctr`: 59,393 (33.6%)
- `other`: 116,668 (66.0%)
- `position_opportunity`: 475 (0.3%)
- `stale_visible_page`: 202 (0.1%)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. Ready to Go.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. Ready to Go.


In [2]:
print("=" * 50)
print("BASELINE RULE DEFINITION")
print("=" * 50)

print("""
Rule: A page is worth reviewing if it has decent impressions, low CTR, and is old.

Score = impressions_90d * (1 - ctr_90d) * (content_age_days / 365)

Reason codes:
  - high_impressions_low_ctr: impressions > 500, ctr < 0.01
  - stale_visible_page: content_age_days > 365, impressions > 500
  - position_opportunity: avg_position > 10, impressions > 500
  - other: none of the above
""")

BASELINE RULE DEFINITION

Rule: A page is worth reviewing if it has decent impressions, low CTR, and is old.

Score = impressions_90d * (1 - ctr_90d) * (content_age_days / 365)

Reason codes:
  - high_impressions_low_ctr: impressions > 500, ctr < 0.01
  - stale_visible_page: content_age_days > 365, impressions > 500
  - position_opportunity: avg_position > 10, impressions > 500
  - other: none of the above



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


**Steps:**
1. Load feature vector (auto-build if missing)
2. Fill missing values (ctr → 0, avg_position → 10, content_age → 0)
3. Calculate baseline score
4. Assign reason codes
5. Sort by score descending
6. Write to `work/outputs/baseline_action_score.csv`

**Results:**
- Queue written: 176,738 rows
- Score range: 0 to 628,208
- Top reason code: `high_impressions_low_ctr` (59,393 rows)

In [1]:
import pandas as pd
import os
import duckdb
from google.colab import userdata

print("=" * 50)
print("LOADING / BUILDING FEATURE VECTOR")
print("=" * 50)

cache_path = 'work/outputs/feature_vector_march2026.parquet'

# Check if cached file exists
if os.path.exists(cache_path):
    print("Loading cached feature vector...")
    feature_vector = pd.read_parquet(cache_path)
    print(f" Loaded: {len(feature_vector):,} rows, {len(feature_vector.columns)} columns")
else:
    print("Cache not found. Building feature vector from warehouse...")

    # Connect and authenticate
    con = duckdb.connect()
    HF_TOKEN = userdata.get('HF_TOKEN')
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

    REL = 'hf://datasets/FlyRank/internship-warehouse'

    # Build feature vector — one row per page
    feature_vector = con.sql(f"""
        WITH daily_features AS (
            SELECT
                d.content_hash_id,
                d.client_hash_id,
                SUM(d.gsc_impressions) AS impressions_90d,
                SUM(d.gsc_clicks) AS clicks_90d,
                AVG(d.gsc_avg_position) AS avg_position_90d,
                CASE
                    WHEN SUM(d.gsc_impressions) > 0
                    THEN SUM(d.gsc_clicks) * 1.0 / SUM(d.gsc_impressions)
                    ELSE 0
                END AS ctr_90d,
                COUNT(DISTINCT d.report_date) AS days_active,
                SUM(d.ga4_sessions) AS sessions_90d,
                SUM(d.ga4_engaged_sessions) AS engaged_sessions_90d
            FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet') d
            WHERE d.gsc_impressions IS NOT NULL AND d.gsc_impressions > 0
            GROUP BY d.content_hash_id, d.client_hash_id
        )
        SELECT
            d.*,
            DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days,
            c.content_type,
            c.word_count,
            c.search_volume,
            c.main_intent
        FROM daily_features d
        LEFT JOIN read_parquet('{REL}/dim_content.parquet') c
            ON d.content_hash_id = c.content_hash_id
        WHERE c.content_created_date IS NOT NULL
    """).df()

    print(f" Built: {len(feature_vector):,} rows, {len(feature_vector.columns)} columns")

    # Save to cache
    os.makedirs('work/outputs', exist_ok=True)
    feature_vector.to_parquet(cache_path)
    print(f" Cached to {cache_path}")

print("\n" + "=" * 50)
print("FILLING MISSING VALUES")
print("=" * 50)

feature_vector['ctr_90d'] = feature_vector['ctr_90d'].fillna(0)
feature_vector['avg_position_90d'] = feature_vector['avg_position_90d'].fillna(10)
feature_vector['content_age_days'] = feature_vector['content_age_days'].fillna(0)

print("Filled: ctr_90d, avg_position_90d, content_age_days")

# Calculate baseline score
print("\n" + "=" * 50)
print("CALCULATING BASELINE SCORE")
print("=" * 50)

feature_vector['baseline_score'] = (
    feature_vector['impressions_90d'] *
    (1 - feature_vector['ctr_90d']) *
    (feature_vector['content_age_days'] / 365)
)

print(f"Score range: {feature_vector['baseline_score'].min():.2f} to {feature_vector['baseline_score'].max():.2f}")

# Assign reason codes
print("\n" + "=" * 50)
print("ASSIGNING REASON CODES")
print("=" * 50)

def assign_reason(row):
    if row['impressions_90d'] > 500 and row['ctr_90d'] < 0.01:
        return 'high_impressions_low_ctr'
    elif row['content_age_days'] > 365 and row['impressions_90d'] > 500:
        return 'stale_visible_page'
    elif row['avg_position_90d'] > 10 and row['impressions_90d'] > 500:
        return 'position_opportunity'
    else:
        return 'other'

feature_vector['reason_code'] = feature_vector.apply(assign_reason, axis=1)

print(feature_vector['reason_code'].value_counts())

# Rank and write CSV
print("\n" + "=" * 50)
print("RANKING AND WRITING CSV")
print("=" * 50)

baseline_queue = feature_vector.sort_values('baseline_score', ascending=False)

# Select key columns for CSV
queue_output = baseline_queue[['content_hash_id', 'client_hash_id', 'baseline_score',
                               'reason_code', 'impressions_90d', 'ctr_90d',
                               'content_age_days', 'avg_position_90d']].copy()

# Write to CSV
os.makedirs('work/outputs', exist_ok=True)
queue_output.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f" Queue written: {len(queue_output):,} rows")
print(f"   File: work/outputs/baseline_action_score.csv")
print(f"\nTop 5 rows:")
print(queue_output.head(5))

LOADING / BUILDING FEATURE VECTOR
Cache not found. Building feature vector from warehouse...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 Built: 176,738 rows, 14 columns
 Cached to work/outputs/feature_vector_march2026.parquet

FILLING MISSING VALUES
Filled: ctr_90d, avg_position_90d, content_age_days

CALCULATING BASELINE SCORE
Score range: 0.00 to 628208.22

ASSIGNING REASON CODES
reason_code
other                       116668
high_impressions_low_ctr     59393
position_opportunity           475
stale_visible_page             202
Name: count, dtype: int64

RANKING AND WRITING CSV
 Queue written: 176,738 rows
   File: work/outputs/baseline_action_score.csv

Top 5 rows:
                 content_hash_id           client_hash_id  baseline_score  \
24139   content_eadb33b5df496f4a  client_e547b89c05043229   628208.219178   
113330  content_ec2e0346994fb5a5  client_e547b89c05043229   289883.463014   
24140   content_0e03de7680314cd5  client_e547b89c05043229   226633.561644   
24110   content_8d7d99f109e19aa2  client_e547b89c05043229   208775.342466   
24137   content_4ffe18112a5642e3  client_e547b89c05043229   191503.767123

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

**Key observations:**
- 19 of top 20 flagged as `high_impressions_low_ctr`
- 1 flagged as `other` (rank 6)
- All have high impressions (97K–617K) and low CTR (0–1.2%)

**What would make these wrong:**
- CTR increases naturally due to seasonality or brand demand
- CTR is normal for the content category/intent
- Pages already in top positions — SERP features may explain low CTR

**Confidence:** High for most — but check seasonal patterns before acting.

In [2]:
print("=" * 50)
print("TOP 20 REVIEW")
print("=" * 50)

# Load the queue if not already loaded
if 'baseline_queue' not in locals():
    baseline_queue = pd.read_csv('work/outputs/baseline_action_score.csv')

top20 = baseline_queue.head(20)

actions = {
    'high_impressions_low_ctr': 'Refresh content',
    'stale_visible_page': 'Update content',
    'position_opportunity': 'Improve SEO',
    'other': 'Monitor'
}

print(f"{'Rank':<6} {'Action':<20} {'Reason Code':<25} {'Impressions':<12} {'CTR':<10} {'Age':<8}")
print("-" * 85)

for i, (idx, row) in enumerate(top20.iterrows(), 1):
    action = actions.get(row['reason_code'], 'Monitor')
    print(f"{i:<6} {action:<20} {row['reason_code']:<25} {row['impressions_90d']:<12.0f} {row['ctr_90d']:<10.4f} {row['content_age_days']:<8.0f}")

print("\n" + "=" * 50)
print("What would make these wrong?")
print("=" * 50)

for i, (idx, row) in enumerate(top20.iterrows(), 1):
    if row['reason_code'] == 'high_impressions_low_ctr':
        print(f"{i}. If CTR increases naturally due to seasonality or brand demand")
    elif row['reason_code'] == 'stale_visible_page':
        print(f"{i}. If the page is intentionally left unchanged (evergreen content)")
    else:
        print(f"{i}. If the page is already performing well for its intent")

TOP 20 REVIEW
Rank   Action               Reason Code               Impressions  CTR        Age     
-------------------------------------------------------------------------------------
1      Refresh content      high_impressions_low_ctr  617124       0.0092     375     
2      Refresh content      high_impressions_low_ctr  245276       0.0060     434     
3      Refresh content      high_impressions_low_ctr  221310       0.0033     375     
4      Refresh content      high_impressions_low_ctr  203497       0.0014     375     
5      Refresh content      high_impressions_low_ctr  186983       0.0031     375     
6      Monitor              other                     205045       0.0119     343     
7      Refresh content      high_impressions_low_ctr  151166       0.0027     410     
8      Refresh content      high_impressions_low_ctr  164885       0.0024     375     
9      Refresh content      high_impressions_low_ctr  142304       0.0024     412     
10     Refresh content      hi

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


**Weak picks identified:**

1. **All top 20 are one type:** 19 of 20 flagged as `high_impressions_low_ctr`. This suggests the rule is too narrowly defined — consider adding diversity (e.g., content_type weighting).

2. **Pages already in top positions:** Many top picks have avg_position ~2-3. Low CTR may be due to SERP features, not content quality.

3. **Seasonal CTR issues:** Low CTR may be seasonal, not a content problem.

**Leakage check:**
-  No product decision flags used
-  No future-window data used
-  No label-derived columns used
-  All features knowable BEFORE decision point

**Implication:** The rule is clean and will serve as a strong baseline. The ML model in Week 5 must beat this baseline.

In [3]:
print("=" * 50)
print("WEAK PICKS + LEAKAGE CHECK")
print("=" * 50)

print("Leakage check:")
print("  - trend_pct: NOT used")
print("  - trend_direction: NOT used")
print("  - Product flags: NOT used")
print("  - Future windows: NOT used")
print("  - Label-derived: NOT used")
print("\n All features are knowable BEFORE the decision point.")

print("\n" + "=" * 50)
print("BASELINE STATISTICS")
print("=" * 50)

print(f"Total pages scored: {len(baseline_queue):,}")
print(f"Pages with reason_code other: {len(baseline_queue[baseline_queue['reason_code'] == 'other']):,}")
print(f"Pages with reason_code high_impressions_low_ctr: {len(baseline_queue[baseline_queue['reason_code'] == 'high_impressions_low_ctr']):,}")
print(f"Pages with reason_code position_opportunity: {len(baseline_queue[baseline_queue['reason_code'] == 'position_opportunity']):,}")
print(f"Pages with reason_code stale_visible_page: {len(baseline_queue[baseline_queue['reason_code'] == 'stale_visible_page']):,}")

print("\n" + "=" * 50)
print("WEAK PICKS IN TOP 20")
print("=" * 50)

weak_picks = [
    "All top 20 are high_impressions_low_ctr — consider adding diversity to rule",
    "Some pages already in top positions — CTR may be normal for their category",
    "Low CTR may be seasonal, not a content quality issue"
]

for i, pick in enumerate(weak_picks, 1):
    print(f"{i}. {pick}")

print("\n" + "=" * 50)
print("NEXT STEPS")
print("=" * 50)
print("""
1. Baseline precision@K will be computed
2. ML model in Week 5 must beat this baseline
3. Compare model results to baseline on same slice
4. Consider adding content_type weighting to improve rule diversity
""")

WEAK PICKS + LEAKAGE CHECK
Leakage check:
  - trend_pct: NOT used
  - trend_direction: NOT used
  - Product flags: NOT used
  - Future windows: NOT used
  - Label-derived: NOT used

 All features are knowable BEFORE the decision point.

BASELINE STATISTICS
Total pages scored: 176,738
Pages with reason_code other: 116,668
Pages with reason_code high_impressions_low_ctr: 59,393
Pages with reason_code position_opportunity: 475
Pages with reason_code stale_visible_page: 202

WEAK PICKS IN TOP 20
1. All top 20 are high_impressions_low_ctr — consider adding diversity to rule
2. Some pages already in top positions — CTR may be normal for their category
3. Low CTR may be seasonal, not a content quality issue

NEXT STEPS

1. Baseline precision@K will be computed
2. ML model in Week 5 must beat this baseline
3. Compare model results to baseline on same slice
4. Consider adding content_type weighting to improve rule diversity



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.